In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime



Index(['timestamp', 'itemid', 'property', 'value'], dtype='object')

In [5]:
concat = pd.concat([
    pd.read_csv('../data/item_properties_part1.csv', usecols=['itemid', 'property', 'value', 'timestamp']), 
    pd.read_csv('../data/item_properties_part2.csv', usecols=['itemid', 'property', 'value', 'timestamp'])
])
cat_tree = pd.read_csv('../data/category_tree.csv')
events = pd.read_csv('../data/events.csv')
item_props_sorted = concat.sort_values(by = 'timestamp', ascending= True)
item_props_filtered  = item_props_sorted.drop_duplicates(['itemid', 'property'], keep = 'last')
item_props_filtered = item_props_filtered[item_props_filtered['property'].isin(['categoryid', 'available'])]

item_props_pivot = item_props_filtered.pivot(index= 'itemid', columns = 'property', values= 'value').reset_index()
item_props_pivot['categoryid'] = pd.to_numeric(item_props_pivot['categoryid'], errors='coerce')

props_category = pd.merge(
    item_props_pivot, 
    cat_tree, 
    on= 'categoryid', 
    how= 'left'
)

df = props_category[['itemid', 'categoryid', 'parentid', 'available']].merge(
    events, 
    on= 'itemid', 
    how = 'right'
)


In [9]:
df.to_csv('../data/processed/events_enriched.csv', index=False)
df.isnull().sum()

itemid                 0
categoryid        255585
parentid          255592
available         255585
timestamp              0
visitorid              0
event                  0
transactionid    2733644
dtype: int64

,itemid,categoryid,parentid,available,timestamp,visitorid,event,transactionid
0,355908,1173.0,805.0,1,1433221332117,257597,view,NaN
1,248676,1231.0,901.0,1,1433224214164,992329,view,NaN
2,318965,NaN,NaN,NaN,1433221999827,111016,view,NaN
3,253185,914.0,226.0,0,1433221955914,483717,view,NaN
4,367447,491.0,679.0,0,1433221337106,951259,view,NaN


(2756101, 8)